# KKBox Churn Prediction -- Exploratory Data Analysis

**Dataset**: WSDM Cup 2018 KKBox Churn Prediction Challenge (v2 refresh)

**Goal**: Understand the structure, quality, and distributions of the four source tables
before building features and models.

**Tables**:
- `train_v2.csv` -- target labels (msno, is_churn)
- `members_v3.csv` -- user demographics
- `transactions_v2.csv` -- payment/subscription history
- `user_logs_v2.csv` -- daily listening behavior

**Churn definition**: A user churned if they did not renew their subscription within
30 days after their membership expired.

In [ ]:
import sys
from pathlib import Path

# Allow imports from project root when running the notebook
sys.path.insert(0, str(Path.cwd().parent))

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from config.paths import (
    TRAIN_PATH, MEMBERS_PATH, TRANSACTIONS_PATH,
    USER_LOGS_PATH, FIGURES_DIR,
)

# Ensure output directory exists
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Shared DuckDB connection (in-memory, no persistent file)
con = duckdb.connect()

# Plot style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 120

print("Setup complete.")

## 1. Data Inventory

Schema, row counts, and column types for each table. All reads go through DuckDB
so we never load multi-GB files into pandas memory.

In [ ]:
tables = {
    "train_v2": TRAIN_PATH,
    "members_v3": MEMBERS_PATH,
    "transactions_v2": TRANSACTIONS_PATH,
    "user_logs_v2": USER_LOGS_PATH,
}

inventory = []
for name, path in tables.items():
    row_count = con.execute(
        f"SELECT COUNT(*) AS n FROM read_csv_auto('{path}')"
    ).fetchone()[0]
    schema = con.execute(
        f"DESCRIBE SELECT * FROM read_csv_auto('{path}')"
    ).fetchdf()
    columns = ", ".join(f"{r['column_name']} ({r['column_type']})" for _, r in schema.iterrows())
    inventory.append({
        "Table": name,
        "Rows": f"{row_count:,}",
        "Columns": schema.shape[0],
        "Schema": columns,
    })

pd.DataFrame(inventory)

## 2. Target Distribution -- Class Imbalance

The competition labels users as churned (1) or retained (0). Severe imbalance
affects model training, threshold selection, and metric choice.

In [ ]:
churn_dist = con.execute(f"""
    SELECT
        is_churn,
        COUNT(*) AS n,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM read_csv_auto('{TRAIN_PATH}')
    GROUP BY is_churn
    ORDER BY is_churn
""").fetchdf()

print(churn_dist.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    churn_dist["is_churn"].astype(str),
    churn_dist["n"],
    color=["#4C72B0", "#DD8452"],
    edgecolor="white",
)
for bar, pct in zip(bars, churn_dist["pct"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 5000,
        f"{pct}%", ha="center", va="bottom", fontsize=12, fontweight="bold",
    )
ax.set_xlabel("is_churn")
ax.set_ylabel("Number of users")
ax.set_title("Target Distribution: Churn vs Retained")
ax.set_xticklabels(["Retained (0)", "Churned (1)"])
fig.tight_layout()
fig.savefig(FIGURES_DIR / "01_churn_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Missing Value Audit

The members table has known quality issues: the `bd` (age) field contains outliers
(negative values, values over 100), and `gender` is frequently empty. This audit
quantifies how much cleaning is needed before feature engineering.

In [ ]:
# -- Members table: missing / invalid values --
members_quality = con.execute(f"""
    SELECT
        COUNT(*) AS total_members,
        SUM(CASE WHEN gender IS NULL OR gender = '' THEN 1 ELSE 0 END) AS gender_missing,
        ROUND(SUM(CASE WHEN gender IS NULL OR gender = '' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS gender_missing_pct,
        SUM(CASE WHEN bd <= 0 OR bd > 100 THEN 1 ELSE 0 END) AS age_invalid,
        ROUND(SUM(CASE WHEN bd <= 0 OR bd > 100 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS age_invalid_pct,
        SUM(CASE WHEN city IS NULL THEN 1 ELSE 0 END) AS city_missing,
        SUM(CASE WHEN registered_via IS NULL THEN 1 ELSE 0 END) AS reg_via_missing,
        SUM(CASE WHEN registration_init_time IS NULL THEN 1 ELSE 0 END) AS reg_time_missing
    FROM read_csv_auto('{MEMBERS_PATH}')
""").fetchdf()

print("=== Members Table Quality ===")
print(members_quality.T.to_string(header=False))

In [ ]:
# -- Transactions & user_logs: missing value check --
txn_quality = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN payment_method_id IS NULL THEN 1 ELSE 0 END) AS pay_method_null,
        SUM(CASE WHEN transaction_date IS NULL THEN 1 ELSE 0 END) AS txn_date_null,
        SUM(CASE WHEN membership_expire_date IS NULL THEN 1 ELSE 0 END) AS expire_null,
        SUM(CASE WHEN actual_amount_paid IS NULL THEN 1 ELSE 0 END) AS amount_null
    FROM read_csv_auto('{TRANSACTIONS_PATH}')
""").fetchdf()

logs_quality = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN total_secs IS NULL THEN 1 ELSE 0 END) AS total_secs_null,
        SUM(CASE WHEN num_unq IS NULL THEN 1 ELSE 0 END) AS num_unq_null,
        SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS date_null
    FROM read_csv_auto('{USER_LOGS_PATH}')
""").fetchdf()

print("=== Transactions Quality ===")
print(txn_quality.T.to_string(header=False))
print("\n=== User Logs Quality ===")
print(logs_quality.T.to_string(header=False))

In [ ]:
# -- Age distribution (valid range only: 10-70) --
age_df = con.execute(f"""
    SELECT bd AS age
    FROM read_csv_auto('{MEMBERS_PATH}')
    WHERE bd BETWEEN 10 AND 70
""").fetchdf()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Valid ages histogram
axes[0].hist(age_df["age"], bins=60, color="#4C72B0", edgecolor="white")
axes[0].set_title("Age Distribution (valid: 10-70)")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Count")

# All bd values (to show the outlier problem)
all_bd = con.execute(f"""
    SELECT bd FROM read_csv_auto('{MEMBERS_PATH}')
""").fetchdf()
axes[1].hist(all_bd["bd"], bins=100, color="#DD8452", edgecolor="white", log=True)
axes[1].set_title("Raw bd field (log scale, showing outliers)")
axes[1].set_xlabel("bd value")
axes[1].set_ylabel("Count (log)")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_age_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Valid ages: {len(age_df):,} / {len(all_bd):,} ({len(age_df)/len(all_bd)*100:.1f}%)")

## 4. Churn Rate by Demographics

Joining train labels with member attributes to see which segments churn more.
These patterns guide feature engineering and help sanity-check model outputs later.

In [ ]:
# -- Churn rate by gender --
churn_gender = con.execute(f"""
    SELECT
        CASE WHEN m.gender IN ('male', 'female') THEN m.gender ELSE 'unknown' END AS gender,
        COUNT(*) AS n,
        ROUND(AVG(t.is_churn) * 100, 2) AS churn_rate
    FROM read_csv_auto('{TRAIN_PATH}') t
    JOIN read_csv_auto('{MEMBERS_PATH}') m ON t.msno = m.msno
    GROUP BY 1
    ORDER BY n DESC
""").fetchdf()

# -- Churn rate by city (top 15) --
churn_city = con.execute(f"""
    SELECT
        m.city,
        COUNT(*) AS n,
        ROUND(AVG(t.is_churn) * 100, 2) AS churn_rate
    FROM read_csv_auto('{TRAIN_PATH}') t
    JOIN read_csv_auto('{MEMBERS_PATH}') m ON t.msno = m.msno
    GROUP BY m.city
    HAVING COUNT(*) > 1000
    ORDER BY n DESC
    LIMIT 15
""").fetchdf()

# -- Churn rate by registration channel --
churn_reg = con.execute(f"""
    SELECT
        m.registered_via,
        COUNT(*) AS n,
        ROUND(AVG(t.is_churn) * 100, 2) AS churn_rate
    FROM read_csv_auto('{TRAIN_PATH}') t
    JOIN read_csv_auto('{MEMBERS_PATH}') m ON t.msno = m.msno
    GROUP BY m.registered_via
    HAVING COUNT(*) > 500
    ORDER BY n DESC
""").fetchdf()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Gender
bars = axes[0].bar(churn_gender["gender"], churn_gender["churn_rate"], color="#4C72B0", edgecolor="white")
axes[0].set_title("Churn Rate by Gender")
axes[0].set_ylabel("Churn Rate (%)")
for bar, val in zip(bars, churn_gender["churn_rate"]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f"{val}%", ha="center", fontsize=9)

# City
axes[1].barh(churn_city["city"].astype(str), churn_city["churn_rate"], color="#55A868", edgecolor="white")
axes[1].set_title("Churn Rate by City (top 15)")
axes[1].set_xlabel("Churn Rate (%)")
axes[1].invert_yaxis()

# Registration channel
bars = axes[2].bar(churn_reg["registered_via"].astype(str), churn_reg["churn_rate"],
                   color="#C44E52", edgecolor="white")
axes[2].set_title("Churn Rate by Registration Channel")
axes[2].set_ylabel("Churn Rate (%)")
for bar, val in zip(bars, churn_reg["churn_rate"]):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f"{val}%", ha="center", fontsize=9)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "03_churn_by_demographics.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Registration Cohort Analysis

Does tenure affect churn? Plotting churn rate by registration month
reveals whether newer or older users are more likely to leave.

In [ ]:
# registration_init_time is an integer like 20110911 (YYYYMMDD)
# Extract year-month for cohort grouping
cohort_churn = con.execute(f"""
    SELECT
        CAST(m.registration_init_time / 100 AS INTEGER) AS reg_month,
        COUNT(*) AS n,
        ROUND(AVG(t.is_churn) * 100, 2) AS churn_rate
    FROM read_csv_auto('{TRAIN_PATH}') t
    JOIN read_csv_auto('{MEMBERS_PATH}') m ON t.msno = m.msno
    WHERE m.registration_init_time IS NOT NULL
      AND m.registration_init_time > 20040101
    GROUP BY 1
    HAVING COUNT(*) > 100
    ORDER BY 1
""").fetchdf()

# Convert YYYYMM integer to datetime for plotting
cohort_churn["date"] = pd.to_datetime(cohort_churn["reg_month"].astype(str), format="%Y%m")

fig, ax1 = plt.subplots(figsize=(14, 5))

color_rate = "#C44E52"
color_count = "#4C72B0"

ax1.plot(cohort_churn["date"], cohort_churn["churn_rate"], color=color_rate, linewidth=1.5)
ax1.set_xlabel("Registration Month")
ax1.set_ylabel("Churn Rate (%)", color=color_rate)
ax1.tick_params(axis="y", labelcolor=color_rate)
ax1.set_title("Churn Rate and User Count by Registration Cohort")

# Overlay user count as bars on secondary axis
ax2 = ax1.twinx()
ax2.bar(cohort_churn["date"], cohort_churn["n"], width=25, alpha=0.25, color=color_count)
ax2.set_ylabel("Users in cohort", color=color_count)
ax2.tick_params(axis="y", labelcolor=color_count)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "04_cohort_churn_rate.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Transaction Patterns

Subscription and payment behavior is one of the strongest signal groups for churn.
Key things to look at: auto-renew status, cancellation flags, plan pricing,
and whether users received discounts.

In [ ]:
# -- Basic transaction stats --
txn_stats = con.execute(f"""
    SELECT
        COUNT(*) AS total_transactions,
        COUNT(DISTINCT msno) AS unique_users,
        ROUND(AVG(payment_plan_days), 1) AS avg_plan_days,
        ROUND(AVG(plan_list_price), 1) AS avg_list_price,
        ROUND(AVG(actual_amount_paid), 1) AS avg_amount_paid,
        ROUND(AVG(CASE WHEN actual_amount_paid < plan_list_price THEN 1.0 ELSE 0.0 END) * 100, 1) AS discount_pct,
        ROUND(AVG(is_auto_renew) * 100, 1) AS auto_renew_pct,
        ROUND(AVG(is_cancel) * 100, 1) AS cancel_pct
    FROM read_csv_auto('{TRANSACTIONS_PATH}')
""").fetchdf()

print("=== Transaction Summary ===")
print(txn_stats.T.to_string(header=False))

In [ ]:
# -- Churn rate by auto-renew and cancel status --
# Use the LAST transaction per user to represent their current state
churn_by_txn = con.execute(f"""
    WITH last_txn AS (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY msno ORDER BY transaction_date DESC) AS rn
        FROM read_csv_auto('{TRANSACTIONS_PATH}')
    )
    SELECT
        lt.is_auto_renew,
        lt.is_cancel,
        COUNT(*) AS n,
        ROUND(AVG(t.is_churn) * 100, 2) AS churn_rate
    FROM read_csv_auto('{TRAIN_PATH}') t
    JOIN last_txn lt ON t.msno = lt.msno AND lt.rn = 1
    GROUP BY lt.is_auto_renew, lt.is_cancel
    ORDER BY lt.is_auto_renew, lt.is_cancel
""").fetchdf()

print("Churn rate by (is_auto_renew, is_cancel):")
print(churn_by_txn.to_string(index=False))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Auto-renew
ar = churn_by_txn.groupby("is_auto_renew").agg({"n": "sum", "churn_rate": "mean"}).reset_index()
bars = axes[0].bar(ar["is_auto_renew"].astype(str), ar["churn_rate"],
                   color=["#DD8452", "#4C72B0"], edgecolor="white")
axes[0].set_xticklabels(["No auto-renew", "Auto-renew"])
axes[0].set_ylabel("Churn Rate (%)")
axes[0].set_title("Churn Rate by Auto-Renew Status")
for bar, val in zip(bars, ar["churn_rate"]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{val:.1f}%", ha="center", fontsize=10)

# Payment plan days distribution
plan_days = con.execute(f"""
    SELECT payment_plan_days, COUNT(*) AS n
    FROM read_csv_auto('{TRANSACTIONS_PATH}')
    GROUP BY payment_plan_days
    ORDER BY n DESC
    LIMIT 10
""").fetchdf()
axes[1].bar(plan_days["payment_plan_days"].astype(str), plan_days["n"],
            color="#55A868", edgecolor="white")
axes[1].set_xlabel("Plan Duration (days)")
axes[1].set_ylabel("Transaction Count")
axes[1].set_title("Most Common Plan Durations")
axes[1].tick_params(axis="x", rotation=45)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "05_transaction_patterns.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# -- Discount analysis and churn rate by payment method --
churn_discount = con.execute(f"""
    WITH last_txn AS (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY msno ORDER BY transaction_date DESC) AS rn
        FROM read_csv_auto('{TRANSACTIONS_PATH}')
    )
    SELECT
        CASE WHEN lt.actual_amount_paid < lt.plan_list_price THEN 'Discounted' ELSE 'Full price' END AS price_type,
        COUNT(*) AS n,
        ROUND(AVG(t.is_churn) * 100, 2) AS churn_rate
    FROM read_csv_auto('{TRAIN_PATH}') t
    JOIN last_txn lt ON t.msno = lt.msno AND lt.rn = 1
    GROUP BY 1
""").fetchdf()

churn_pay_method = con.execute(f"""
    WITH last_txn AS (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY msno ORDER BY transaction_date DESC) AS rn
        FROM read_csv_auto('{TRANSACTIONS_PATH}')
    )
    SELECT
        lt.payment_method_id,
        COUNT(*) AS n,
        ROUND(AVG(t.is_churn) * 100, 2) AS churn_rate
    FROM read_csv_auto('{TRAIN_PATH}') t
    JOIN last_txn lt ON t.msno = lt.msno AND lt.rn = 1
    GROUP BY lt.payment_method_id
    HAVING COUNT(*) > 500
    ORDER BY n DESC
    LIMIT 12
""").fetchdf()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Discount vs full price
bars = axes[0].bar(churn_discount["price_type"], churn_discount["churn_rate"],
                   color=["#4C72B0", "#DD8452"], edgecolor="white")
axes[0].set_ylabel("Churn Rate (%)")
axes[0].set_title("Churn Rate: Discounted vs Full Price")
for bar, val in zip(bars, churn_discount["churn_rate"]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{val}%", ha="center", fontsize=10)

# Payment method
axes[1].barh(churn_pay_method["payment_method_id"].astype(str),
             churn_pay_method["churn_rate"], color="#8172B2", edgecolor="white")
axes[1].set_xlabel("Churn Rate (%)")
axes[1].set_title("Churn Rate by Payment Method (top 12)")
axes[1].invert_yaxis()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "06_discount_payment_churn.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Transaction Temporal Patterns

Number of transactions per user and days-until-expiry from the last transaction.
Users with more transaction history and longer remaining subscriptions may behave
differently from those about to expire.

In [ ]:
# -- Transactions per user + days until expiry --
txn_per_user = con.execute(f"""
    SELECT
        t.is_churn,
        txn.txn_count,
        txn.days_until_expiry
    FROM read_csv_auto('{TRAIN_PATH}') t
    JOIN (
        SELECT
            msno,
            COUNT(*) AS txn_count,
            DATEDIFF('day',
                STRPTIME(MAX(transaction_date)::VARCHAR, '%Y%m%d'),
                STRPTIME(MAX(membership_expire_date)::VARCHAR, '%Y%m%d')
            ) AS days_until_expiry
        FROM read_csv_auto('{TRANSACTIONS_PATH}')
        GROUP BY msno
    ) txn ON t.msno = txn.msno
""").fetchdf()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Transactions per user by churn status
for label, color in [(0, "#4C72B0"), (1, "#DD8452")]:
    subset = txn_per_user[txn_per_user["is_churn"] == label]["txn_count"].clip(upper=20)
    axes[0].hist(subset, bins=20, alpha=0.6, color=color,
                 label=f"{'Churned' if label else 'Retained'}", edgecolor="white")
axes[0].set_xlabel("Number of Transactions (capped at 20)")
axes[0].set_ylabel("Users")
axes[0].set_title("Transactions per User by Churn Status")
axes[0].legend()

# Days until expiry
for label, color in [(0, "#4C72B0"), (1, "#DD8452")]:
    subset = txn_per_user[txn_per_user["is_churn"] == label]["days_until_expiry"]
    subset = subset[(subset > -100) & (subset < 400)]  # filter extreme outliers
    axes[1].hist(subset, bins=50, alpha=0.6, color=color,
                 label=f"{'Churned' if label else 'Retained'}", edgecolor="white")
axes[1].set_xlabel("Days Until Expiry (from last transaction)")
axes[1].set_ylabel("Users")
axes[1].set_title("Days Until Expiry by Churn Status")
axes[1].legend()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "07_transaction_temporal.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Median txn count -- retained: {txn_per_user[txn_per_user['is_churn']==0]['txn_count'].median():.0f}, "
      f"churned: {txn_per_user[txn_per_user['is_churn']==1]['txn_count'].median():.0f}")
print(f"Median days_until_expiry -- retained: {txn_per_user[txn_per_user['is_churn']==0]['days_until_expiry'].median():.0f}, "
      f"churned: {txn_per_user[txn_per_user['is_churn']==1]['days_until_expiry'].median():.0f}")

## 8. User Listening Behavior

The user_logs table (18.3M rows, 1.3GB) contains daily listening activity.
DuckDB aggregates this efficiently without loading the full file into pandas.

Columns:
- `num_25/50/75/985/100`: count of songs played to 25%/50%/75%/98.5%/100% completion
- `num_unq`: unique songs played that day
- `total_secs`: total listening time in seconds

In [ ]:
# -- Check date range and basic stats --
logs_range = con.execute(f"""
    SELECT
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT msno) AS unique_users
    FROM read_csv_auto('{USER_LOGS_PATH}')
""").fetchdf()

print("=== User Logs Date Range ===")
print(logs_range.T.to_string(header=False))

In [ ]:
# -- Per-user listening aggregates, joined with churn labels --
user_activity = con.execute(f"""
    SELECT
        t.is_churn,
        logs.active_days,
        logs.total_secs,
        logs.total_complete_plays,
        logs.total_unique_songs,
        logs.total_partial_plays
    FROM read_csv_auto('{TRAIN_PATH}') t
    JOIN (
        SELECT
            msno,
            COUNT(DISTINCT date) AS active_days,
            SUM(total_secs) AS total_secs,
            SUM(num_100) AS total_complete_plays,
            SUM(num_unq) AS total_unique_songs,
            SUM(num_25) AS total_partial_plays
        FROM read_csv_auto('{USER_LOGS_PATH}')
        GROUP BY msno
    ) logs ON t.msno = logs.msno
""").fetchdf()

print(f"Users with listening data: {len(user_activity):,}")
print(f"\nSummary by churn status:")
print(user_activity.groupby("is_churn")[
    ["active_days", "total_secs", "total_unique_songs"]
].median().round(1).to_string())

In [ ]:
# -- Behavioral distributions: churners vs retained --
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
metrics = [
    ("active_days", "Active Days", axes[0, 0], None),
    ("total_secs", "Total Listening Seconds", axes[0, 1], 500_000),
    ("total_unique_songs", "Unique Songs Played", axes[1, 0], 3000),
    ("total_complete_plays", "Fully Completed Plays", axes[1, 1], 5000),
]

for col, title, ax, clip_val in metrics:
    for label, color in [(0, "#4C72B0"), (1, "#DD8452")]:
        subset = user_activity[user_activity["is_churn"] == label][col].dropna()
        if clip_val:
            subset = subset.clip(upper=clip_val)
        ax.hist(subset, bins=50, alpha=0.6, color=color,
                label=f"{'Churned' if label else 'Retained'}", edgecolor="white", density=True)
    ax.set_title(title)
    ax.set_ylabel("Density")
    ax.legend()

fig.suptitle("Listening Behavior: Churners vs Retained Users", fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "08_listening_behavior.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# -- Engagement depth: ratio of unique songs to partial plays (num_25) --
# Higher ratio means the user explores more music rather than skipping
engagement = user_activity.copy()
engagement["engagement_depth"] = (
    engagement["total_unique_songs"] / engagement["total_partial_plays"].replace(0, float("nan"))
)

fig, ax = plt.subplots(figsize=(8, 4))
for label, color in [(0, "#4C72B0"), (1, "#DD8452")]:
    subset = engagement[engagement["is_churn"] == label]["engagement_depth"].dropna().clip(upper=5)
    ax.hist(subset, bins=50, alpha=0.6, color=color,
            label=f"{'Churned' if label else 'Retained'}", edgecolor="white", density=True)
ax.set_xlabel("Engagement Depth (unique_songs / partial_plays)")
ax.set_ylabel("Density")
ax.set_title("Engagement Depth: Churners vs Retained")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "09_engagement_depth.png", dpi=150, bbox_inches="tight")
plt.show()

print("Median engagement depth:")
print(engagement.groupby("is_churn")["engagement_depth"].median())

## 9. Correlation Heatmap

A quick look at pairwise correlations between the numeric features we have explored
so far. This previews which signals are redundant and which are independently
informative for predicting churn.

In [ ]:
# -- Build a mini feature table for correlation analysis --
mini_features = con.execute(f"""
    SELECT
        t.is_churn,
        -- Member features
        CASE WHEN m.bd BETWEEN 10 AND 70 THEN m.bd ELSE NULL END AS age,
        DATEDIFF('day',
            STRPTIME(m.registration_init_time::VARCHAR, '%Y%m%d'),
            DATE '2017-03-01'
        ) AS tenure_days,
        -- Transaction features (last transaction)
        lt.is_auto_renew AS last_auto_renew,
        lt.is_cancel AS last_cancel,
        lt.payment_plan_days AS last_plan_days,
        CASE WHEN lt.actual_amount_paid < lt.plan_list_price THEN 1 ELSE 0 END AS has_discount,
        txn_agg.txn_count,
        txn_agg.days_until_expiry,
        -- Listening features
        logs.active_days,
        logs.total_secs,
        logs.total_unique_songs
    FROM read_csv_auto('{TRAIN_PATH}') t
    LEFT JOIN read_csv_auto('{MEMBERS_PATH}') m ON t.msno = m.msno
    LEFT JOIN (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY msno ORDER BY transaction_date DESC) AS rn
        FROM read_csv_auto('{TRANSACTIONS_PATH}')
    ) lt ON t.msno = lt.msno AND lt.rn = 1
    LEFT JOIN (
        SELECT msno, COUNT(*) AS txn_count,
            DATEDIFF('day',
                STRPTIME(MAX(transaction_date)::VARCHAR, '%Y%m%d'),
                STRPTIME(MAX(membership_expire_date)::VARCHAR, '%Y%m%d')
            ) AS days_until_expiry
        FROM read_csv_auto('{TRANSACTIONS_PATH}')
        GROUP BY msno
    ) txn_agg ON t.msno = txn_agg.msno
    LEFT JOIN (
        SELECT msno,
            COUNT(DISTINCT date) AS active_days,
            SUM(total_secs) AS total_secs,
            SUM(num_unq) AS total_unique_songs
        FROM read_csv_auto('{USER_LOGS_PATH}')
        GROUP BY msno
    ) logs ON t.msno = logs.msno
""").fetchdf()

print(f"Mini feature table: {mini_features.shape[0]:,} rows x {mini_features.shape[1]} columns")
print(f"\nNull counts:\n{mini_features.isnull().sum().to_string()}")

In [ ]:
# -- Correlation heatmap --
corr = mini_features.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
mask = [[False]*len(corr.columns) for _ in range(len(corr))]
for i in range(len(corr)):
    for j in range(i):
        mask[i][j] = True  # mask lower triangle for cleaner display

sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
    square=True, linewidths=0.5, ax=ax,
    vmin=-1, vmax=1,
    cbar_kws={"shrink": 0.8},
)
ax.set_title("Feature Correlation Matrix")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "10_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# Correlation with churn target specifically
print("Correlation with is_churn (sorted by absolute value):")
churn_corr = corr["is_churn"].drop("is_churn").abs().sort_values(ascending=False)
print(churn_corr.to_string())

## 10. Key Findings

### Class Imbalance
- The dataset has a severe class imbalance (expected ~94% retained / ~6% churned).
- Implication: accuracy is a misleading metric. We will use ROC-AUC, PR-AUC, and F1.
  For modeling, LightGBM's `scale_pos_weight` parameter will handle this.

### Data Quality
- **Age (`bd`)**: Large share of invalid values (0, negative, or > 100). Will be set to NULL
  for values outside the 10-70 range.
- **Gender**: Frequently missing/empty. Will be encoded as a third "unknown" category.
- **Transactions and user_logs**: Generally clean with minimal nulls.

### Strongest Churn Signals (from EDA)
1. **`is_auto_renew`**: Users without auto-renew churn at dramatically higher rates.
   This is expected (no auto-renew means the user must actively decide to stay).
2. **`is_cancel`**: Direct cancellation flag correlates strongly with churn.
3. **Active days / listening time**: Churners tend to have less recent activity.
4. **Tenure**: The cohort analysis reveals how registration vintage affects churn.
5. **Transaction count**: More historical transactions signal stickier users.

### Features to Engineer (Stage 2)
- Transaction group: last payment method, auto-renew, cancel, plan days, discount flag,
  total transactions, days until expiry
- Listening group: 30-day aggregates of active days, seconds played, unique songs,
  engagement depth (unique/partial ratio)
- Member group: age (cleaned), city, gender, registration channel, tenure in days

### Tradeoff: user_logs date window
The user_logs_v2 data appears to cover a specific month (to be confirmed by the date
range check above). If it only covers one month, the "30-day aggregate" is effectively
the entire log dataset, which simplifies feature engineering but limits our ability to
compute multi-window features (7d vs 14d vs 30d). This will be addressed in Stage 2.